# MintPy on HyP3 Products

User-friendly workflow for preparing ASF HyP3 Sentinel-1 GeoTIFF products for MintPy, running `smallbaselineApp.py`, and locating the outputs needed by downstream `snowsar` notebooks.

## Quick start

1. Activate the `snowsar` environment and open this notebook.
2. Update `HYP3_DIR` in the configuration cell.
3. Run all cells from top to bottom.
4. If preflight fails, fix the exact item listed and rerun.
5. Use `final_timeseries_h5` in downstream notebooks such as `MintPy_SNOTEL.ipynb`.

## What this notebook assumes

- Your HyP3 products are already downloaded and extracted.
- Each interferogram lives in one folder, or all files are under one common directory tree.
- Required inputs include unwrapped phase, correlation, DEM, and HyP3 metadata `.txt` files.
- Incidence angle files are recommended for geometry-aware downstream analysis.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap

import pandas as pd
from IPython.display import Markdown, display

from snowsar.utils import (
    build_insar_context,
    clip_hyp3_products_to_common_overlap,
    find_matching_products,
    parse_date_pairs_from_hyp3_filenames,
    plot_hyp3_trio,
)


## 1. Configuration

Edit this cell only for a normal run.


In [ ]:
# Directory containing extracted HyP3 product folders.
HYP3_DIR = Path("/Volumes/Fortress_L3/SnowWaterEquivalent/SWE_Shadi/HyP3/hyp3_data/data")

# MintPy working directory. Template and MintPy outputs are written here.
MINTPY_WORK_DIR = HYP3_DIR / "mintpy"

# Set True to create common-overlap clipped rasters before MintPy loading.
CLIP_TO_COMMON_OVERLAP = True

# Set True to run smallbaselineApp.py after writing the MintPy template.
RUN_MINTPY = True

# Index of the interferogram to preview in the quick visual check cell.
IFG_INDEX = 0


## 2. Advanced Configuration

Leave these defaults alone unless you are debugging or reusing an existing clipped stack.


In [ ]:
OVERWRITE_CLIPPED = False
USE_CLIPPED_PRODUCTS = True
RUN_PREP_HYP3_FIRST = False
TEMPLATE_FILE = MINTPY_WORK_DIR / "smallbaselineApp.cfg"

RAW_PATTERNS = {
    "unwrapped_phase": "*unw_phase.tif",
    "correlation": "*corr.tif",
    "dem": "*dem.tif",
    "incidence_angle": "*lv_theta.tif",
    "azimuth_angle": "*lv_phi.tif",
    "water_mask": "*water_mask.tif",
    "wrapped_phase": "*wrapped_phase.tif",
}

CLIPPED_PATTERNS = {
    "unwrapped_phase": "*unw_phase_clip*.tif",
    "correlation": "*corr_clip*.tif",
    "dem": "*dem_clip*.tif",
    "incidence_angle": "*lv_theta_clip*.tif",
    "azimuth_angle": "*lv_phi_clip*.tif",
    "water_mask": "*water_mask_clip*.tif",
    "wrapped_phase": "*wrapped_phase_clip*.tif",
}

ACTIVE_PATTERNS = CLIPPED_PATTERNS if USE_CLIPPED_PRODUCTS else RAW_PATTERNS
REQUIRED_PRODUCT_KEYS = ("unwrapped_phase", "correlation", "dem")
OPTIONAL_PRODUCT_KEYS = ("incidence_angle", "azimuth_angle", "water_mask", "wrapped_phase")


## 3. Helper Functions


In [ ]:
def find_hyp3_files(pattern: str) -> list[Path]:
    return sorted(HYP3_DIR.rglob(pattern, recurse_symlinks=True))


def collect_files(patterns: dict[str, str]) -> dict[str, list[Path]]:
    return {name: find_hyp3_files(pattern) for name, pattern in patterns.items()}


def mintpy_glob(pattern: str) -> str:
    one_level_matches = list(HYP3_DIR.glob(f"*/{pattern}"))
    if one_level_matches:
        return f"{HYP3_DIR.as_posix()}/*/{pattern}"
    return f"{HYP3_DIR.as_posix()}/{pattern}"


def tail_text(text: str, max_lines: int = 30) -> str:
    lines = [line for line in text.splitlines() if line.strip()]
    if not lines:
        return ""
    if len(lines) <= max_lines:
        return "\n".join(lines)
    return "\n".join(["...", *lines[-max_lines:]])


def run_command(command: list[str], cwd: Path, *, preview_lines: int = 30) -> subprocess.CompletedProcess:
    print("$ " + " ".join(command))
    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    combined_output = "\n".join(part for part in (result.stdout, result.stderr) if part)
    preview = tail_text(combined_output, max_lines=preview_lines)

    if result.returncode == 0:
        print(f"Completed successfully in {cwd}")
        if preview:
            print(preview)
        return result

    print("Command failed. Recent output:")
    if preview:
        print(preview)
    raise RuntimeError(
        f"Command failed with exit code {result.returncode}: {' '.join(command)}"
    )


def show_preflight_table(raw_files: dict[str, list[Path]], clipped_files: dict[str, list[Path]], metadata_files: list[Path]) -> None:
    rows = []
    for name in [*REQUIRED_PRODUCT_KEYS, *OPTIONAL_PRODUCT_KEYS]:
        rows.append(
            {
                "product": name,
                "raw_count": len(raw_files.get(name, [])),
                "clipped_count": len(clipped_files.get(name, [])),
                "active_pattern": ACTIVE_PATTERNS[name],
            }
        )
    rows.append(
        {
            "product": "hyp3_metadata",
            "raw_count": len(metadata_files),
            "clipped_count": len(metadata_files),
            "active_pattern": "*.txt",
        }
    )
    display(pd.DataFrame(rows))


def require(condition: bool, message: str, errors: list[str]) -> None:
    if not condition:
        errors.append(message)


## 4. Preflight Check

This cell tells you whether the notebook is ready to run before any processing starts.


In [ ]:
if str(HYP3_DIR) == "/path/to/extracted/hyp3_products":
    raise ValueError("Update HYP3_DIR in the configuration cell before running the notebook.")

if not HYP3_DIR.exists():
    raise FileNotFoundError(
        f"HyP3 directory not found: {HYP3_DIR}\n"
        "Update HYP3_DIR to point to your extracted HyP3 product directory."
    )

MINTPY_WORK_DIR.mkdir(parents=True, exist_ok=True)

raw_files = collect_files(RAW_PATTERNS)
clipped_files = collect_files(CLIPPED_PATTERNS)
metadata_files = find_hyp3_files("*.txt")
active_files = collect_files(ACTIVE_PATTERNS)

print(f"HyP3 directory: {HYP3_DIR}")
print(f"MintPy work directory: {MINTPY_WORK_DIR}")
print(f"Active input mode: {'clipped products' if USE_CLIPPED_PRODUCTS else 'raw products'}")
print(f"Clipping step enabled: {CLIP_TO_COMMON_OVERLAP}")
print(f"MintPy run enabled: {RUN_MINTPY}")
show_preflight_table(raw_files, clipped_files, metadata_files)

errors: list[str] = []
warnings: list[str] = []

if CLIP_TO_COMMON_OVERLAP:
    for key in REQUIRED_PRODUCT_KEYS:
        require(
            len(raw_files[key]) > 0,
            f"Missing raw {key} files required for clipping. Expected pattern: {RAW_PATTERNS[key]}",
            errors,
        )
else:
    for key in REQUIRED_PRODUCT_KEYS:
        require(
            len(active_files[key]) > 0,
            f"Missing {key} files for MintPy. Expected pattern: {ACTIVE_PATTERNS[key]}",
            errors,
        )

require(
    len(metadata_files) > 0,
    "No HyP3 metadata .txt files found. MintPy preparation requires those text files.",
    errors,
)

if len(active_files["incidence_angle"]) == 0:
    warnings.append(
        "Incidence angle files were not found. MintPy may still run, but downstream SWE conversion will be limited."
    )

smallbaseline_path = shutil.which("smallbaselineApp.py")
prep_hyp3_path = shutil.which("prep_hyp3.py")
print(f"smallbaselineApp.py: {smallbaseline_path or 'not found on PATH'}")
print(f"prep_hyp3.py: {prep_hyp3_path or 'not found on PATH'}")

if RUN_MINTPY:
    require(
        smallbaseline_path is not None,
        "smallbaselineApp.py is not on PATH. Activate the snowsar environment that includes MintPy.",
        errors,
    )
if RUN_PREP_HYP3_FIRST:
    require(
        prep_hyp3_path is not None,
        "prep_hyp3.py is not on PATH. Activate the snowsar environment that includes MintPy.",
        errors,
    )

if warnings:
    display(Markdown("\n".join(f"- Warning: {message}" for message in warnings)))

if errors:
    raise RuntimeError("Preflight failed:\n- " + "\n- ".join(errors))

print("Preflight passed. Continue to clipping or input audit.")


## 5. Optional Common-Overlap Clipping

Run this only when `CLIP_TO_COMMON_OVERLAP = True`. If you already have clipped rasters, the notebook skips this step.


In [ ]:
if CLIP_TO_COMMON_OVERLAP:
    clipped_outputs = clip_hyp3_products_to_common_overlap(
        HYP3_DIR,
        dem_pattern="*/*_dem.tif",
        overwrite=OVERWRITE_CLIPPED,
    )
    print(f"Prepared {len(clipped_outputs)} clipped products on the common overlap.")
else:
    print("Skipping clipping. Notebook will use files that already match the active patterns.")


## 6. Audit Active Inputs

This cell refreshes the file lists that MintPy will actually use.


In [ ]:
active_files = collect_files(ACTIVE_PATTERNS)
unw_files = active_files["unwrapped_phase"]
cor_files = active_files["correlation"]
dem_files = active_files["dem"]
inc_files = active_files["incidence_angle"]
az_files = active_files["azimuth_angle"]
water_files = active_files["water_mask"]
wrapped_files = active_files["wrapped_phase"]
metadata_files = find_hyp3_files("*.txt")

counts = pd.DataFrame(
    [
        {"product": "unwrapped_phase", "pattern": ACTIVE_PATTERNS["unwrapped_phase"], "count": len(unw_files)},
        {"product": "correlation", "pattern": ACTIVE_PATTERNS["correlation"], "count": len(cor_files)},
        {"product": "dem", "pattern": ACTIVE_PATTERNS["dem"], "count": len(dem_files)},
        {"product": "incidence_angle", "pattern": ACTIVE_PATTERNS["incidence_angle"], "count": len(inc_files)},
        {"product": "azimuth_angle", "pattern": ACTIVE_PATTERNS["azimuth_angle"], "count": len(az_files)},
        {"product": "water_mask", "pattern": ACTIVE_PATTERNS["water_mask"], "count": len(water_files)},
        {"product": "wrapped_phase", "pattern": ACTIVE_PATTERNS["wrapped_phase"], "count": len(wrapped_files)},
        {"product": "hyp3_metadata", "pattern": "*.txt", "count": len(metadata_files)},
    ]
)
display(counts)

if not unw_files:
    raise FileNotFoundError(
        f"No unwrapped phase files found with {ACTIVE_PATTERNS['unwrapped_phase']}. "
        "If you expected clipped files, enable CLIP_TO_COMMON_OVERLAP or switch USE_CLIPPED_PRODUCTS to False."
    )
if not cor_files:
    raise FileNotFoundError(f"No correlation files found with {ACTIVE_PATTERNS['correlation']}.")
if not dem_files:
    raise FileNotFoundError(f"No DEM files found with {ACTIVE_PATTERNS['dem']}.")
if IFG_INDEX < 0 or IFG_INDEX >= len(unw_files):
    raise IndexError(f"IFG_INDEX={IFG_INDEX} is out of range. Valid range: 0 to {len(unw_files) - 1}.")

date_pairs = parse_date_pairs_from_hyp3_filenames(unw_files)
pairs_df = pd.DataFrame(
    {
        "reference_date": [ref.date() for ref, _ in date_pairs],
        "secondary_date": [sec.date() for _, sec in date_pairs],
        "temporal_baseline_days": [(sec - ref).days for ref, sec in date_pairs],
    }
)
print(f"Parsed {len(date_pairs)} interferogram date pairs.")
display(pairs_df.head(20))


## 7. Build HyP3 Context


In [ ]:
hyp3_ctx = build_insar_context(source="hyp3", hyp3_tifs=unw_files)

print(f"Source: {hyp3_ctx.source}")
print(f"Acquisition dates: {len(hyp3_ctx.dates)}")
print(f"Date range: {hyp3_ctx.dates[0].date()} to {hyp3_ctx.dates[-1].date()}")
print(f"Footprint CRS: {hyp3_ctx.footprint.crs}")
display(hyp3_ctx.footprint)


## 8. Quick Visual Check

Use this cell to confirm one interferogram looks reasonable before running MintPy.


In [ ]:
import matplotlib.pyplot as plt

unw_file = unw_files[IFG_INDEX]
wrapped_file, corr_file = find_matching_products(unw_file, wrapped_files, cor_files)

print(f"Preview index: {IFG_INDEX}")
print(f"Unwrapped: {unw_file.name}")
print(f"Wrapped:   {wrapped_file.name if wrapped_file else 'not found'}")
print(f"Coherence: {corr_file.name if corr_file else 'not found'}")

fig, axes = plot_hyp3_trio(
    unw_file,
    wrapped_file,
    corr_file,
    unw_vlim_rad=None,
    corr_vlim=(0, 1),
    figsize=(18, 7),
)
plt.show()


## 9. Write MintPy Template


In [ ]:
unw_glob = mintpy_glob(ACTIVE_PATTERNS["unwrapped_phase"])
cor_glob = mintpy_glob(ACTIVE_PATTERNS["correlation"])
dem_glob = mintpy_glob(ACTIVE_PATTERNS["dem"])
inc_glob = mintpy_glob(ACTIVE_PATTERNS["incidence_angle"]) if inc_files else "no"
az_glob = mintpy_glob(ACTIVE_PATTERNS["azimuth_angle"]) if az_files else "no"
water_glob = mintpy_glob(ACTIVE_PATTERNS["water_mask"]) if water_files else "no"

template_text = textwrap.dedent(
    f"""
    ########## snowsar HyP3 -> MintPy template
    ## Generated by notebooks/codex_mintpy_hyp3.ipynb

    ########## 1. load_data
    mintpy.load.processor        = hyp3
    mintpy.load.autoPath         = no
    mintpy.load.updateMode       = yes
    mintpy.load.compression      = auto

    ##---------interferogram datasets:
    mintpy.load.unwFile          = {unw_glob}
    mintpy.load.corFile          = {cor_glob}
    mintpy.load.connCompFile     = no

    ##---------geometry datasets:
    mintpy.load.demFile          = {dem_glob}
    mintpy.load.incAngleFile     = {inc_glob}
    mintpy.load.azAngleFile      = {az_glob}
    mintpy.load.waterMaskFile    = {water_glob}

    ########## keep first run local and reproducible
    mintpy.troposphericDelay.method = no
    mintpy.deramp               = auto
    mintpy.topographicResidual  = auto
    """
).strip() + "\n"

TEMPLATE_FILE.write_text(template_text)
print(f"Wrote template: {TEMPLATE_FILE}")
print(template_text)


## 10. Run MintPy

The first cell optionally prepares HyP3 metadata. The second cell runs `smallbaselineApp.py`.


In [ ]:
if RUN_PREP_HYP3_FIRST:
    prep_inputs = [unw_files, cor_files, dem_files, inc_files, az_files, water_files]
    for files in prep_inputs:
        if files:
            run_command(["prep_hyp3.py", *[str(path) for path in files]], cwd=MINTPY_WORK_DIR)
else:
    print("Skipping prep_hyp3.py. MintPy will prepare HyP3 metadata during load_data.")


In [ ]:
if RUN_MINTPY:
    run_command(["smallbaselineApp.py", TEMPLATE_FILE.name], cwd=MINTPY_WORK_DIR)
else:
    print("RUN_MINTPY is False. Run this command manually when ready:")
    print(f"cd {MINTPY_WORK_DIR}")
    print(f"smallbaselineApp.py {TEMPLATE_FILE.name}")


## 11. Inspect MintPy Outputs


In [ ]:
from mintpy.utils import readfile

timeseries_candidates = sorted(
    set(MINTPY_WORK_DIR.glob("timeseries*.h5")) | set(MINTPY_WORK_DIR.glob("geo_timeseries*.h5"))
)
geometry_candidates = sorted(
    set(MINTPY_WORK_DIR.glob("geometry*.h5"))
    | set(MINTPY_WORK_DIR.glob("geo_geometry*.h5"))
    | set((MINTPY_WORK_DIR / "inputs").glob("geometry*.h5"))
)

if not timeseries_candidates:
    raise FileNotFoundError(f"No MintPy timeseries*.h5 output found in {MINTPY_WORK_DIR}")

final_timeseries_h5 = max(timeseries_candidates, key=lambda path: path.stat().st_mtime)
attrs = readfile.read_attribute(str(final_timeseries_h5))

summary = pd.DataFrame(
    [
        {"kind": "timeseries", "path": str(path), "modified": pd.to_datetime(path.stat().st_mtime, unit="s")}
        for path in timeseries_candidates
    ]
    + [
        {"kind": "geometry", "path": str(path), "modified": pd.to_datetime(path.stat().st_mtime, unit="s")}
        for path in geometry_candidates
    ]
)
display(summary.sort_values(["kind", "path"]).reset_index(drop=True))

print(f"Selected final time series: {final_timeseries_h5}")
display(
    pd.DataFrame(
        [
            {
                "FILE_TYPE": attrs.get("FILE_TYPE"),
                "LENGTH": attrs.get("LENGTH"),
                "WIDTH": attrs.get("WIDTH"),
                "X_FIRST": attrs.get("X_FIRST"),
                "Y_FIRST": attrs.get("Y_FIRST"),
                "X_STEP": attrs.get("X_STEP"),
                "Y_STEP": attrs.get("Y_STEP"),
                "X_UNIT": attrs.get("X_UNIT"),
                "Y_UNIT": attrs.get("Y_UNIT"),
                "EPSG": attrs.get("EPSG"),
            }
        ]
    )
)


## 12. Build Downstream Context

This produces the objects and paths used by other notebooks.


In [ ]:
mintpy_ctx = build_insar_context(
    source="mintpy",
    mintpy_timeseries_h5=final_timeseries_h5,
    mintpy_reference_slice=None,
)

print(f"MintPy context dates: {len(mintpy_ctx.dates)}")
print(f"MintPy context date range: {mintpy_ctx.dates[0].date()} to {mintpy_ctx.dates[-1].date()}")
display(mintpy_ctx.footprint)

print("Next files to use:")
print(f"  MintPy time series: {final_timeseries_h5}")
if geometry_candidates:
    print(f"  Geometry candidates: {[str(path) for path in geometry_candidates]}")
else:
    print("  Geometry candidates: none found")


## What Next

- Open `MintPy_SNOTEL.ipynb` if you want station-based validation.
- Use `final_timeseries_h5` as the MintPy displacement input for downstream `snowsar` analysis.
- Reuse `MINTPY_WORK_DIR` if you want to rerun MintPy with a modified template.
